# Trening modelu segmentacji Kraken na polskich stronach EHRI

Fine-tuning domyslnego modelu segmentacji Kraken (blla) na 15 polskich stronach EHRI.

**Cel:** poprawic CER z 14,25% (domyslny segmenter) do ~11% (lepsza segmentacja).

**Metoda:**
1. Pobierz 15 polskich stron .tif + ALTO XML z EHRI
2. Podzial: 12 stron train, 3 strony validation
3. Fine-tune domyslnego modelu segmentacji z ketos segtrain --load
4. Ewaluacja: Kraken e2e z custom segmenter vs domyslny

**Uwaga:** 15 stron to malo - uzywamy fine-tuningu (nie from scratch) + augmentacji.

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = '4cdc2bf9edb8bd23ece7042a8e89e82a9ad34db8'
EHRI_DATASET_REPO = 'PiotrSty/ehri-dataset'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
# Kraken 7.1 wymaga nowszej huggingface_hub (is_offline_mode)
subprocess.run([sys.executable,'-m','pip','install','kraken>=7.0','jiwer','pillow','opencv-python-headless','albumentations','huggingface_hub>=0.23'],check=True)
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.'):
        del sys.modules[_mod]
from importlib.metadata import version as _pkg_version
print('IMPORT_OK', 'kraken', _pkg_version('kraken'), 'huggingface_hub', _pkg_version('huggingface_hub'))

In [ ]:
import torch, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
assert torch.cuda.is_available(), "GPU required; select Kaggle GPU T4."
print("GPU:", torch.cuda.get_device_name(0))

ehri_dir = Path("/kaggle/working/ehri-polish")
ehri_dir.mkdir(parents=True, exist_ok=True)
all_files = list_repo_files(EHRI_DATASET_REPO, repo_type="dataset")
polish_files = [f for f in all_files if "polish" in f and (f.endswith(".tif") or f.endswith(".xml"))]
for f in polish_files:
    path = hf_hub_download(EHRI_DATASET_REPO, f, repo_type="dataset")
    shutil.copy(path, ehri_dir / Path(f).name)

kraken_model_path = hf_hub_download(EHRI_DATASET_REPO, "models/polish_nfd_9313.mlmodel", repo_type="dataset")
print("Kraken recognition model:", kraken_model_path)
print("Polish pages:", len(list(ehri_dir.glob("*.tif"))))
print("ALTO XML:", len(list(ehri_dir.glob("*.xml"))))


In [ ]:
import random
random.seed(42)
pages = sorted(ehri_dir.glob("*.tif"))
random.shuffle(pages)
val_pages = pages[:3]
train_pages = pages[3:]
print(f"Train: {len(train_pages)} pages")
for p in train_pages:
    print(f"  {p.name}")
print(f"Validation: {len(val_pages)} pages")
for p in val_pages:
    print(f"  {p.name}")

train_manifest = ehri_dir / "train_manifest.txt"
val_manifest = ehri_dir / "val_manifest.txt"
train_manifest.write_text("\n".join(str(p.with_suffix(".xml")) for p in train_pages))
val_manifest.write_text("\n".join(str(p.with_suffix(".xml")) for p in val_pages))
print(f"\nManifests written:")
print(f"  Train: {train_manifest}")
print(f"  Val: {val_manifest}")


In [ ]:
# Znajdź domyślny model segmentacji Kraken (lokalny, wbudowany w pakiet)
import kraken
from pathlib import Path
kraken_dir = Path(kraken.__file__).parent
seg_models = list(kraken_dir.rglob('*.safetensors')) + list(kraken_dir.rglob('*.mlmodel'))
print('Kraken dir:', kraken_dir)
print('Segmentation models found:')
for m in seg_models:
    print(f'  {m}')

# Użyj lokalnego modelu blla.mlmodel (wbudowany w pakiet Kraken)
default_seg = str(kraken_dir / 'blla.mlmodel') if (kraken_dir / 'blla.mlmodel').exists() else None
print(f'\nDefault seg model (local): {default_seg}')

In [ ]:
# Trening: fine-tune domyślnego modelu segmentacji (Kraken 7.1)
import subprocess, shutil

output_model = '/kaggle/working/polish_seg'
ketos_bin = shutil.which('ketos') or 'ketos'

# Kraken 7.1: -d i --workers są globalne (przed subkomendą)
# -o to ścieżka wyjściowa (prefix), wynik to .safetensors
cmd = [
    ketos_bin,
    '-d', 'cuda',
    '--workers', '2',
    'segtrain',
    '-f', 'xml',
    '-t', str(train_manifest),
    '-e', str(val_manifest),
    '--augment',
    '-o', output_model,
    '-N', '50',
]

if default_seg:
    cmd += ['-i', default_seg, '--resize', 'new']

print('Training command:')
print(' '.join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT:', result.stdout[-5000:])
print('STDERR:', result.stderr[-5000:])
print('Return code:', result.returncode)

# Znajdź wytrenowany model (safetensors lub mlmodel)
import glob
trained = glob.glob('/kaggle/working/polish_seg*.safetensors') + glob.glob('/kaggle/working/polish_seg*.mlmodel')
print('\nTrained model files:', trained)

In [ ]:
# Ewaluacja: Kraken e2e z custom segmenter vs domyslny
import jiwer, warnings, glob
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
import kraken.lib.models as models
from kraken.blla import segment
from kraken.rpred import rpred

warnings.filterwarnings('ignore')
ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def load_page_gt_from_alto(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        if text:
            lines.append(text)
    return lines

recognizer = models.load_any(kraken_model_path, device='cuda')
ehri_dir = Path('/kaggle/working/ehri-polish')
val_xmls = [ehri_dir / p.with_suffix('.xml').name for p in val_pages]

# Znajdź wytrenowany model
trained = (glob.glob('/kaggle/working/polish_seg*.safetensors')
           + glob.glob('/kaggle/working/polish_seg*.mlmodel')
           + glob.glob('/kaggle/working/*_best.safetensors'))
custom_seg = trained[0] if trained else None
print(f'Custom segmenter: {custom_seg}')

for seg_name, seg_model in [
    ('default', None),
    ('custom', custom_seg),
]:
    if seg_model is None and seg_name == 'custom':
        print(f'\n=== {seg_name}: model not found, skipping ===')
        continue
    print(f'\n=== Kraken e2e + {seg_name} segmenter ===')
    all_refs, all_hyps = [], []
    for xml_path in val_xmls:
        page_path = xml_path.with_suffix('.tif')
        gt_lines = load_page_gt_from_alto(xml_path)
        img = Image.open(page_path).convert('L')
        if seg_model:
            seg = segment(img, device='cuda', model=seg_model)
        else:
            seg = segment(img, device='cuda')
        pred = rpred(recognizer, img, seg)
        hyp_lines = [record.prediction.strip() for record in pred]
        ref = '\n'.join(gt_lines)
        hyp = '\n'.join(hyp_lines)
        all_refs.append(ref)
        all_hyps.append(hyp)
        print(f'  {page_path.name}: {len(gt_lines)} GT, {len(hyp_lines)} OCR')
    if all_refs:
        cer = jiwer.cer(all_refs, all_hyps)
        wer = jiwer.wer(all_refs, all_hyps)
        print(f'  CER: {cer:.4f}  ({cer*100:.2f}%)')
        print(f'  WER: {wer:.4f}  ({wer*100:.2f}%)')